<a href="https://colab.research.google.com/github/lnghan/MC311-Analysis/blob/main/1-ingestion/notebooks/le_capstone_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Han Le
# DATA 205 Project Ingestion Part



---



In [ ]:
# Upload all datasets (MC311, Weather, ZIP Code Population, 8 Datasets on ZIP Code Income Info)

from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime



---



#**Cleaning main dataset (MC311)**

In [ ]:
mc311 = pd.read_csv('mc311_jan2025-apr2026.csv')

In [ ]:
mc311.head()

,SR ID,Opened,Closed,Status,Department,Area,Sub-Area,Attached Solution (Topic),Attached Solution SLA Days,City,...,Congressional District,Congressional Member,Council District,Council Member Name,Changed Date,# of Days Open,Within SLA Windows,SLA Yes,SLA No,Request Type
0,1583111101,01/01/2025 12:39:49 AM,01/03/2025 10:08:51 AM,Closed,HHS,SEPH,Emergency Services,First Month's Rent or Security Deposit - HHS,3.0,Damascus,...,6.0,DAVID TRONE (Democrat),7.0,Dawn Luedtke,01/03/2025 03:08:53 PM,2,Yes,1,0,Service Request - Fulfillment
1,1583125811,01/01/2025 06:38:59 AM,01/10/2025 09:07:14 AM,Closed,DEP,Solid Waste,Bulk Trash Request,Bulk Trash Pick-Up Request,5.0,Derwood,...,6.0,DAVID TRONE (Democrat),7.0,Dawn Luedtke,01/10/2025 02:07:14 PM,6,No,0,1,Service Request - Fulfillment
2,1583125821,01/01/2025 06:41:25 AM,01/10/2025 09:08:11 AM,Closed,DEP,Solid Waste,Scrap Metal Request,Scrap Metal Pick-Up Request,5.0,Brookeville,...,3.0,JOHN P. SARBANES (Democrat),7.0,Dawn Luedtke,01/10/2025 02:08:13 PM,6,No,0,1,Service Request - Fulfillment
3,1583125802,01/01/2025 07:54:32 AM,01/01/2025 10:19:40 AM,Closed,POL,Animal Services,Dead Animal SHA,Reporting a Dead Animal Along the Roadway or o...,1.0,Dickerson,...,6.0,DAVID TRONE (Democrat),2.0,Marilyn Balcombe,01/01/2025 03:19:41 PM,0,Yes,1,0,Service Request - Fulfillment
4,1583125841,01/01/2025 08:01:30 AM,01/03/2025 01:14:56 PM,Closed,DEP,Solid Waste,Scrap Metal Request,Scrap Metal Pick-Up Request,5.0,Silver Spring,...,8.0,JAMIE RASKIN. (Democrat),4.0,Kate Stewart,01/03/2025 06:14:56 PM,2,Yes,1,0,Service Request - Fulfillment


In [ ]:
##Exploring the "Area" variable and each of their corresponding departments so I can choose which 3 I would like to focus on

area_department = mc311.groupby(['Area', 'Department']).size().nlargest(10)
display(area_department)

,,0
Area,Department,
Solid Waste,DEP,221392
Treasury,FIN,43450
Highway Services,DOT,35808
Transit,DOT,32387
Children Youth and Families,HHS,29438
CSC,TEBS,27623
Residential Construction,DPS,21693
State of Maryland,Non-MCG,19061
Tree Maintenance,DOT,16429


In [ ]:
# Narrowing the data down to 5 randomly chosen ZIP codes and viewing the counts for each "Area"

specific_zip_codes = [20833, 20866, 20904, 20906, 20853]
explore_mc311 = mc311[mc311['Zip Code'].isin(specific_zip_codes)]
explore_mc311 = explore_mc311.groupby(['Area', 'Department']).size().nlargest(10)
display(explore_mc311)

,,0
Area,Department,
Solid Waste,DEP,37349
Highway Services,DOT,6209
Code Enforcement,DHCA,2709
Tree Maintenance,DOT,2436
Treasury,FIN,1479
Licensing and Registration,DHCA,1365
Landlord Tenant Affairs,DHCA,1158
Animal Services,POL,1150
Traffic,DOT,970


##Intrestingly, when filtered to the randomized ZIP codes, the ranking of the counts of each "Area" changes drastically. This could mean that some issues are more or less prevalent in certain regions of Montgomery County.

In [ ]:
# Using SQL to filter chosen Areas and ZIP Codes

conn = sqlite3.connect('mc311.db')
c = conn.cursor()

In [ ]:
mc311.to_sql('requests', conn, if_exists='replace', index=False)

675250

In [ ]:
conn = sqlite3.connect('mc311.db')
query = '''
SELECT *
FROM requests
WHERE Area IN ('Highway Services', 'Code Enforcement', 'Treasury')
AND "Zip Code" IN ('20833', '20866', '20904', '20906', '20853')
'''

mc311_clean = pd.read_sql(query, conn)
display(mc311_clean)

conn.close()

,SR ID,Opened,Closed,Status,Department,Area,Sub-Area,Attached Solution (Topic),Attached Solution SLA Days,City,...,Congressional District,Congressional Member,Council District,Council Member Name,Changed Date,# of Days Open,Within SLA Windows,SLA Yes,SLA No,Request Type
0,1583125970,01/01/2025 10:17:09 AM,01/07/2025 09:35:57 AM,Closed,DHCA,Code Enforcement,None,Housing Complaints,60.0,Silver Spring,...,3.0,JOHN P. SARBANES (Democrat),5.0,Kristin Mink,01/07/2025 02:35:57 PM,3,Yes,1,0,Service Request - Fulfillment
1,1583126482,01/01/2025 03:33:58 PM,01/15/2025 01:57:34 PM,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Rockville,...,6.0,DAVID TRONE (Democrat),6.0,Natali Fani-Gonz??A!lez,01/15/2025 06:57:37 PM,9,No,0,1,Service Request - Fulfillment
2,1583126709,01/01/2025 04:05:36 PM,01/14/2025 08:48:06 AM,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Silver Spring,...,8.0,JAMIE RASKIN. (Democrat),5.0,Kristin Mink,01/14/2025 01:48:08 PM,8,No,0,1,Service Request - Fulfillment
3,1583126729,01/01/2025 05:16:28 PM,01/02/2025 09:52:41 AM,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Burtonsville,...,3.0,JOHN P. SARBANES (Democrat),5.0,Kristin Mink,01/02/2025 02:52:43 PM,1,Yes,1,0,Service Request - Fulfillment
4,1583154935,01/02/2025 08:32:39 AM,01/07/2025 11:35:01 AM,Closed,DOT,Highway Services,Office,Roadway Resurfacing or Repaving,5.0,Brookeville,...,3.0,JOHN P. SARBANES (Democrat),7.0,Dawn Luedtke,01/07/2025 04:35:05 PM,2,Yes,1,0,Service Request - Fulfillment
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10392,1617015045,04/04/2026 12:05:17 PM,None,In Progress,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,ROCKVILLE,...,8.0,Jamie Raskin,6.0,Natali Fani-Gonz?!lez,04/06/2026 09:04:28 AM,4,No,0,1,Service Request - Fulfillment
10393,1617015089,04/04/2026 12:08:53 PM,None,In Progress,DOT,Highway Services,Road Repair,Road Repair,15.0,SILVER SPRING,...,8.0,Jamie Raskin,5.0,Kristin Mink,04/04/2026 04:10:47 PM,4,Yes,1,0,Service Request - Fulfillment
10394,1617015110,04/04/2026 01:50:42 PM,04/08/2026 06:46:47 AM,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,SILVER SPRING,...,8.0,Jamie Raskin,6.0,Natali Fani-Gonz?!lez,04/08/2026 10:46:49 AM,3,Yes,1,0,Service Request - Fulfillment
10395,1617064269,04/05/2026 01:04:05 PM,04/06/2026 09:50:35 AM,Closed,DHCA,Code Enforcement,None,Housing Complaints,60.0,SILVER SPRING,...,8.0,Jamie Raskin,5.0,Kristin Mink,04/06/2026 01:50:36 PM,1,Yes,1,0,Service Request - Fulfillment


In [ ]:
# Since MC311 dates have time and weather dates do not, I need to fix it in order to merge
mc311_clean['Opened'] = pd.to_datetime(mc311_clean['Opened'],
                                 format='%m/%d/%Y %I:%M:%S %p')

mc311_clean['Closed'] = pd.to_datetime(mc311_clean['Closed'],
                                 format='%m/%d/%Y %I:%M:%S %p')

# Create clean opened and closed columns. Need the opened one for merging and the closed one to calculate response time
mc311_clean['date_opened'] = mc311_clean['Opened'].dt.normalize()
mc311_clean['date_closed'] = mc311_clean['Closed'].dt.normalize()

mc311_clean['response_time'] = (mc311_clean['date_closed'] - mc311_clean['date_opened']).dt.days

# Create a new column 'city_name' with the city names of zip code
zip_to_city = {20904: 'Colesville',
               20906: 'Eastern Silver Spring',
               20866: 'Burtonsville',
               20833: 'Brookeville',
               20853: 'Rockville'}

# Make 'Zip Code' an integer before mapping so it can easily match with dictionary keys
mc311_clean['Zip Code'] = mc311_clean['Zip Code'].astype(int)
mc311_clean['city_name'] = mc311_clean['Zip Code'].map(zip_to_city)

# Ensure city_name is a category
mc311_clean['city_name'] = mc311_clean['city_name'].astype('category')

mc311_clean.head()

,SR ID,Opened,Closed,Status,Department,Area,Sub-Area,Attached Solution (Topic),Attached Solution SLA Days,City,...,Changed Date,# of Days Open,Within SLA Windows,SLA Yes,SLA No,Request Type,date_opened,date_closed,response_time,city_name
0,1583125970,2025-01-01 10:17:09,2025-01-07 09:35:57,Closed,DHCA,Code Enforcement,None,Housing Complaints,60.0,Silver Spring,...,01/07/2025 02:35:57 PM,3,Yes,1,0,Service Request - Fulfillment,2025-01-01,2025-01-07,6.0,Colesville
1,1583126482,2025-01-01 15:33:58,2025-01-15 13:57:34,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Rockville,...,01/15/2025 06:57:37 PM,9,No,0,1,Service Request - Fulfillment,2025-01-01,2025-01-15,14.0,Rockville
2,1583126709,2025-01-01 16:05:36,2025-01-14 08:48:06,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Silver Spring,...,01/14/2025 01:48:08 PM,8,No,0,1,Service Request - Fulfillment,2025-01-01,2025-01-14,13.0,Colesville
3,1583126729,2025-01-01 17:16:28,2025-01-02 09:52:41,Closed,DOT,Highway Services,Pothole Repair,Pothole Repair,3.0,Burtonsville,...,01/02/2025 02:52:43 PM,1,Yes,1,0,Service Request - Fulfillment,2025-01-01,2025-01-02,1.0,Burtonsville
4,1583154935,2025-01-02 08:32:39,2025-01-07 11:35:01,Closed,DOT,Highway Services,Office,Roadway Resurfacing or Repaving,5.0,Brookeville,...,01/07/2025 04:35:05 PM,2,Yes,1,0,Service Request - Fulfillment,2025-01-02,2025-01-07,5.0,Brookeville


In [ ]:
# Keeping variables I want
mc311_clean = mc311_clean[['SR ID',
                           'date_opened',
                           'date_closed',
                           'Status',
                           'response_time',
                           'Attached Solution SLA Days',
                           'Within SLA Windows',
                           'Zip Code',
                           'city_name',
                           'Department',
                           'Area',
                           'Attached Solution (Topic)']]

# Rename variable names for readbility
mc311_clean = mc311_clean.rename(columns={'SR ID': 'id',
                                          'Status': 'status',
                                          'Department': 'department',
                                          'Area': 'area',
                                          'Attached Solution (Topic)': 'area_topic',
                                          'Attached Solution SLA Days': 'sla_days',
                                          'Zip Code': 'zip_code',
                                          'Within SLA Windows': 'within_sla'})

mc311_clean.head()

,id,date_opened,date_closed,status,response_time,sla_days,within_sla,zip_code,city_name,department,area,area_topic
0,1583125970,2025-01-01,2025-01-07,Closed,6.0,60.0,Yes,20904,Colesville,DHCA,Code Enforcement,Housing Complaints
1,1583126482,2025-01-01,2025-01-15,Closed,14.0,3.0,No,20853,Rockville,DOT,Highway Services,Pothole Repair
2,1583126709,2025-01-01,2025-01-14,Closed,13.0,3.0,No,20904,Colesville,DOT,Highway Services,Pothole Repair
3,1583126729,2025-01-01,2025-01-02,Closed,1.0,3.0,Yes,20866,Burtonsville,DOT,Highway Services,Pothole Repair
4,1583154935,2025-01-02,2025-01-07,Closed,5.0,5.0,Yes,20833,Brookeville,DOT,Highway Services,Roadway Resurfacing or Repaving


#**Cleaning weather dataset and merging to main dataset**

In [ ]:
weather = pd.read_csv('md_weather_jan2025-apr2026.csv')
weather.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,TAVG,TAVG_ATTRIBUTES,TMAX,TMAX_ATTRIBUTES,TMIN,TMIN_ATTRIBUTES,WT01,WT01_ATTRIBUTES,WT03,WT03_ATTRIBUTES
0,USW00093784,"MARYLAND SCIENCE CENTER, MD US",39.2814,-76.6111,6.1,2025-01-01,0.00,",,W,2400",NaN,NaN,...,NaN,NaN,51,",,W",42.0,",,W",NaN,NaN,NaN,NaN
1,USW00093784,"MARYLAND SCIENCE CENTER, MD US",39.2814,-76.6111,6.1,2025-01-02,0.00,",,W,2400",NaN,NaN,...,NaN,NaN,42,",,W",37.0,",,W",NaN,NaN,NaN,NaN
2,USW00093784,"MARYLAND SCIENCE CENTER, MD US",39.2814,-76.6111,6.1,2025-01-03,0.02,",,W,2400",NaN,NaN,...,NaN,NaN,43,",,W",32.0,",,W",NaN,NaN,NaN,NaN
3,USW00093784,"MARYLAND SCIENCE CENTER, MD US",39.2814,-76.6111,6.1,2025-01-04,0.00,",,W,2400",NaN,NaN,...,NaN,NaN,34,",,W",27.0,",,W",NaN,NaN,NaN,NaN
4,USW00093784,"MARYLAND SCIENCE CENTER, MD US",39.2814,-76.6111,6.1,2025-01-05,0.00,",,W,2400",NaN,NaN,...,NaN,NaN,34,",,W",24.0,",,W",NaN,NaN,NaN,NaN


In [ ]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   STATION          458 non-null    object 
 1   NAME             458 non-null    object 
 2   LATITUDE         458 non-null    float64
 3   LONGITUDE        458 non-null    float64
 4   ELEVATION        458 non-null    float64
 5   DATE             458 non-null    object 
 6   PRCP             457 non-null    float64
 7   PRCP_ATTRIBUTES  457 non-null    object 
 8   SNOW             4 non-null      float64
 9   SNOW_ATTRIBUTES  4 non-null      object 
 10  SNWD             4 non-null      float64
 11  SNWD_ATTRIBUTES  4 non-null      object 
 12  TAVG             0 non-null      float64
 13  TAVG_ATTRIBUTES  0 non-null      float64
 14  TMAX             458 non-null    int64  
 15  TMAX_ATTRIBUTES  458 non-null    object 
 16  TMIN             457 non-null    float64
 17  TMIN_ATTRIBUTES 

In [ ]:
# Cleaning weather dataset

# Ensure in datetime format for merging
weather['DATE'] = pd.to_datetime(weather['DATE'])

# Select variables I want to keep
weather = weather[['DATE',
                   'PRCP',
                   'TMAX',
                   'TMIN']]

# Change N/As to 0 so it doesn't mess up analyses
weather['PRCP'] = weather['PRCP'].fillna(0)

# Rename variable names for readbility
weather = weather.rename(columns={'DATE': 'date',
                                  'PRCP': 'prcp',
                                  'TMAX': 'max_temp',
                                  'TMIN': 'min_temp'})

# Calculate average temperature manually since the original dataset had the variable, but no dates had any values
weather['avg_temp'] = (weather['max_temp'] + weather['min_temp']) / 2

weather.head()

,date,prcp,max_temp,min_temp,avg_temp
0,2025-01-01,0.00,51,42.0,46.5
1,2025-01-02,0.00,42,37.0,39.5
2,2025-01-03,0.02,43,32.0,37.5
3,2025-01-04,0.00,34,27.0,30.5
4,2025-01-05,0.00,34,24.0,29.0


In [ ]:
merged = pd.merge(mc311_clean, weather, left_on='date_opened', right_on='date', how='left')
merged.head()

,id,date_opened,date_closed,status,response_time,sla_days,within_sla,zip_code,city_name,department,area,area_topic,date,prcp,max_temp,min_temp,avg_temp
0,1583125970,2025-01-01,2025-01-07,Closed,6.0,60.0,Yes,20904,Colesville,DHCA,Code Enforcement,Housing Complaints,2025-01-01,0.0,51.0,42.0,46.5
1,1583126482,2025-01-01,2025-01-15,Closed,14.0,3.0,No,20853,Rockville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5
2,1583126709,2025-01-01,2025-01-14,Closed,13.0,3.0,No,20904,Colesville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5
3,1583126729,2025-01-01,2025-01-02,Closed,1.0,3.0,Yes,20866,Burtonsville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5
4,1583154935,2025-01-02,2025-01-07,Closed,5.0,5.0,Yes,20833,Brookeville,DOT,Highway Services,Roadway Resurfacing or Repaving,2025-01-02,0.0,42.0,37.0,39.5


In [ ]:
# Create "Month" and "Day of the Week" variables for future analyses and exploration
merged['date'] = pd.to_datetime(merged['date'])
merged['day_of_week'] = merged['date'].dt.day_name()
merged['month'] = merged['date'].dt.month_name()

merged.head()

,id,date_opened,date_closed,status,response_time,sla_days,within_sla,zip_code,city_name,department,area,area_topic,date,prcp,max_temp,min_temp,avg_temp,day_of_week,month
0,1583125970,2025-01-01,2025-01-07,Closed,6.0,60.0,Yes,20904,Colesville,DHCA,Code Enforcement,Housing Complaints,2025-01-01,0.0,51.0,42.0,46.5,Wednesday,January
1,1583126482,2025-01-01,2025-01-15,Closed,14.0,3.0,No,20853,Rockville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5,Wednesday,January
2,1583126709,2025-01-01,2025-01-14,Closed,13.0,3.0,No,20904,Colesville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5,Wednesday,January
3,1583126729,2025-01-01,2025-01-02,Closed,1.0,3.0,Yes,20866,Burtonsville,DOT,Highway Services,Pothole Repair,2025-01-01,0.0,51.0,42.0,46.5,Wednesday,January
4,1583154935,2025-01-02,2025-01-07,Closed,5.0,5.0,Yes,20833,Brookeville,DOT,Highway Services,Roadway Resurfacing or Repaving,2025-01-02,0.0,42.0,37.0,39.5,Thursday,January


In [ ]:
# Save the cleaned dataset
merged.to_csv('mc311_weather_clean.csv', index=False)

from google.colab import files
files.download('mc311_weather_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**Cleaning ZIP Codes datasets and merging them [all the ZIP Code datasets] all into one**

In [ ]:
# Dataset with information on socioeconomic factors based on zip code

population = pd.read_csv("zip_codes_population.csv")

population.info()
population.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 468 entries, 0 to 467
Data columns (total 42 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   OBJECTID_1    468 non-null    int64  
 1   ZCTA5CE10     468 non-null    int64  
 2   FIRST_STAT    468 non-null    int64  
 3   FIRST_GEOI    468 non-null    int64  
 4   FIRST_CLAS    468 non-null    object 
 5   FIRST_MTFC    468 non-null    object 
 6   FIRST_FUNC    468 non-null    object 
 7   ZCTA5N        468 non-null    int64  
 8   STATE         468 non-null    int64  
 9   AREALAND      468 non-null    int64  
 10  AREAWATR      468 non-null    int64  
 11  POP100        468 non-null    int64  
 12  HU100         468 non-null    int64  
 13  NHW           468 non-null    int64  
 14  NHB           468 non-null    int64  
 15  NHAI          468 non-null    int64  
 16  NHA           468 non-null    int64  
 17  NHNH          468 non-null    int64  
 18  NHO           468 non-null    

,OBJECTID_1,ZCTA5CE10,FIRST_STAT,FIRST_GEOI,FIRST_CLAS,FIRST_MTFC,FIRST_FUNC,ZCTA5N,STATE,AREALAND,...,VACNS,PVACNS,PHOWN,PWOMORT,PRENT,PLT18SP,REPORT_2_P,REPORT_9_P,Shape_Length,Shape_Area
0,1,20601,24,2420601,B5,G6350,S,20601,24,115635266,...,376,4.3,71.1,11.2,19.9,30.4,http://mdpgis.mdp.state.md.us/Census2010/PDF/0...,http://mdpgis.mdp.state.md.us/census2010/PDF/0...,89500.303353,1.903175e+08
1,2,20602,24,2420602,B5,G6350,S,20602,24,35830723,...,769,7.9,59.7,9.0,34.4,43.6,http://mdpgis.mdp.state.md.us/Census2010/PDF/0...,http://mdpgis.mdp.state.md.us/census2010/PDF/0...,42220.214377,5.930458e+07
2,3,20603,24,2420603,B5,G6350,S,20603,24,44239637,...,531,5.1,73.8,4.7,22.6,29.9,http://mdpgis.mdp.state.md.us/Census2010/PDF/0...,http://mdpgis.mdp.state.md.us/census2010/PDF/0...,50682.962600,7.295967e+07
3,4,20606,24,2420606,B5,G6350,S,20606,24,7501011,...,15,6.5,49.7,39.3,18.1,31.2,http://mdpgis.mdp.state.md.us/Census2010/PDF/0...,http://mdpgis.mdp.state.md.us/census2010/PDF/0...,41458.263845,1.221117e+07
4,5,20607,24,2420607,B5,G6350,S,20607,24,54357590,...,172,4.9,83.1,10.3,7.4,22.1,http://mdpgis.mdp.state.md.us/Census2010/PDF/0...,http://mdpgis.mdp.state.md.us/census2010/PDF/0...,58903.058735,8.962880e+07


In [ ]:
# Filter to selected ZIP codes
population = population[population['ZCTA5CE10'].isin(specific_zip_codes)]

# Keeping select variables
population = population[['ZCTA5CE10',
                         'PLT18SP',
                         'POP100',
                         'HU100',
                         'NHW',
                         'NHB',
                         'NHAI',
                         'NHA',
                         'NHNH',
                         'NHO',
                         'HISP',
                         'POP65_',
                         'MEDAGE']]

# Rename variables
population = population.rename(columns={'ZCTA5CE10': 'zip_code',
                                        'PLT18SP': 'percent_under_18',
                                        'POP100': 'population',
                                        'HU100': 'housing_units',
                                        'NHW': 'white',
                                        'NHB': 'black',
                                        'NHAI': 'american_indian',
                                        'NHA': 'asian',
                                        'NHNH': 'native_hawaiian',
                                        'NHO': 'non_hispanic_other',
                                        'HISP': 'hispanic',
                                        'POP65_': 'population_65_plus',
                                        'MEDAGE': 'med_age'})

population.head()

,zip_code,percent_under_18,population,housing_units,white,black,american_indian,asian,native_hawaiian,non_hispanic_other,hispanic,population_65_plus,med_age
118,20833,12.2,7735,2545,5624,687,6,678,0,30,521,777,42.5
127,20853,19.3,29673,10035,15086,3188,67,3508,10,130,7115,4706,41.4
133,20866,31.9,13344,4652,3483,5703,30,2402,6,59,1287,1022,36.5
151,20904,35.7,54612,21452,14560,22696,81,7533,34,216,8061,8192,38.0
153,20906,34.0,64696,25715,20713,15799,112,7489,32,452,18687,12445,40.1


In [ ]:
# Dataset with information on median income based on zip code

income = pd.read_csv("zip_codes_income.csv")

income.info()
income.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 5 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   placeDcid                       39 non-null     object
 1   placeName                       39 non-null     int64 
 2   Date:Median_Income_Household    39 non-null     int64 
 3   Value:Median_Income_Household   39 non-null     int64 
 4   Source:Median_Income_Household  39 non-null     object
dtypes: int64(3), object(2)
memory usage: 1.7+ KB


,placeDcid,placeName,Date:Median_Income_Household,Value:Median_Income_Household,Source:Median_Income_Household
0,zip/20812,20812,2024,213750,https://www.census.gov/programs-surveys/acs/da...
1,zip/20814,20814,2024,150996,https://www.census.gov/programs-surveys/acs/da...
2,zip/20815,20815,2024,194971,https://www.census.gov/programs-surveys/acs/da...
3,zip/20816,20816,2024,232656,https://www.census.gov/programs-surveys/acs/da...
4,zip/20817,20817,2024,249332,https://www.census.gov/programs-surveys/acs/da...


In [ ]:
# Filter to selected ZIP codes
income = income[income['placeName'].isin(specific_zip_codes)]

# Keeping variables
income = income[['placeName', 'Value:Median_Income_Household']]

# Rename
income = income.rename(columns={'placeName': 'zip_code',
                                'Value:Median_Income_Household': 'med_household_income'})

income.head()

,zip_code,med_household_income
14,20853,159080
20,20866,140308
37,20906,97521


### (This part is pretty long and tedious)

For some reason that dataset I downloaded did not have the median household income for zip codes 20833 and 20904, even though on the same site there are datasets for all income characteristics for those two zip codes. I downloaded all I think are necessary and I will now merge them.

In [ ]:
# Household Median Income

hi_20833 = pd.read_csv("20833_ house_income.csv")
hi_20904 = pd.read_csv("20904_ house_income.csv")

In [ ]:
# FIlter to median income in 2024 only, since original income df was based on 2024 values as well
hi_20833 = hi_20833[hi_20833['Variable observation date'] == 2024]
hi_20904 = hi_20904[hi_20904['Variable observation date'] == 2024]

# Concatenate
household_income = pd.concat([hi_20833, hi_20904], ignore_index=True)

# Keep only needed columns
household_income = household_income[['Entity properties name', 'Variable observation value']]

household_income.head()

,Entity properties name,Variable observation value
0,20833,204375
1,20904,96025


In [ ]:
# Rename columns in household_income to match income to concatenate
household_income = household_income.rename(columns={'Entity properties name': 'zip_code',
                                                    'Variable observation value': 'med_household_income'})

# Concatenate
income = pd.concat([income, household_income], ignore_index=True)

income.head()

,zip_code,med_household_income
0,20853,159080
1,20866,140308
2,20906,97521
3,20833,204375
4,20904,96025


In [ ]:
# Individual Median Income

ii_20853 = pd.read_csv("20853_ indv_income.csv")
ii_20833 = pd.read_csv("20833_ indv_income.csv")
ii_20866 = pd.read_csv("20866_ indv_income.csv")
ii_20904 = pd.read_csv("20904_ indv_income.csv")
ii_20906 = pd.read_csv("20906_ indv_income.csv")

In [ ]:
# Filter to median income in 2024 only, since original income df was based on 2024 values as well
ii_20853 = ii_20853[ii_20853['Variable observation date'] == 2024]
ii_20833 = ii_20833[ii_20833['Variable observation date'] == 2024]
ii_20866 = ii_20866[ii_20866['Variable observation date'] == 2024]
ii_20904 = ii_20904[ii_20904['Variable observation date'] == 2024]
ii_20906 = ii_20906[ii_20906['Variable observation date'] == 2024]

# Concatenate
individual_income = pd.concat([ii_20853, ii_20833, ii_20866, ii_20904, ii_20906], ignore_index=True)

# Keep only needed columns
individual_income = individual_income[['Entity properties name', 'Variable observation value']]

individual_income.head()

,Entity properties name,Variable observation value
0,20853,54311
1,20833,72026
2,20866,53463
3,20904,48018
4,20906,45025


In [ ]:
# Rename columns for merging
individual_income = individual_income.rename(columns={'Entity properties name': 'zip_code',
                                                      'Variable observation value': 'med_individual_income'})

# Merge to income df
income = pd.merge(income, individual_income, on='zip_code', how='left')

income.head()

,zip_code,med_household_income,med_individual_income
0,20853,159080,54311
1,20866,140308,53463
2,20906,97521,45025
3,20833,204375,72026
4,20904,96025,48018


In [ ]:
merged_zips = pd.merge(income, population, on='zip_code', how='left')
merged_zips.head()

,zip_code,med_household_income,med_individual_income,percent_under_18,population,housing_units,white,black,american_indian,asian,native_hawaiian,non_hispanic_other,hispanic,population_65_plus,med_age
0,20853,159080,54311,19.3,29673,10035,15086,3188,67,3508,10,130,7115,4706,41.4
1,20866,140308,53463,31.9,13344,4652,3483,5703,30,2402,6,59,1287,1022,36.5
2,20906,97521,45025,34.0,64696,25715,20713,15799,112,7489,32,452,18687,12445,40.1
3,20833,204375,72026,12.2,7735,2545,5624,687,6,678,0,30,521,777,42.5
4,20904,96025,48018,35.7,54612,21452,14560,22696,81,7533,34,216,8061,8192,38.0


In [166]:
merged_zips['city_name'] = merged_zips['zip_code'].map(zip_to_city)
merged_zips.head()

,zip_code,med_household_income,med_individual_income,percent_under_18,population,housing_units,white,black,american_indian,asian,native_hawaiian,non_hispanic_other,hispanic,population_65_plus,med_age,city_name
0,20853,159080,54311,19.3,29673,10035,15086,3188,67,3508,10,130,7115,4706,41.4,Rockville
1,20866,140308,53463,31.9,13344,4652,3483,5703,30,2402,6,59,1287,1022,36.5,Burtonsville
2,20906,97521,45025,34.0,64696,25715,20713,15799,112,7489,32,452,18687,12445,40.1,Eastern Silver Spring
3,20833,204375,72026,12.2,7735,2545,5624,687,6,678,0,30,521,777,42.5,Brookeville
4,20904,96025,48018,35.7,54612,21452,14560,22696,81,7533,34,216,8061,8192,38.0,Colesville


In [167]:
merged_zips.to_csv('zip_codes_clean.csv', index=False)

from google.colab import files
files.download('zip_codes_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# I don't want to merge zip code data to main dataset yet as I think it will complicate things.